In [1]:
!pip install google-cloud-monitoring


     |████████████████████████████████| 348kB 1.2MB/s eta 0:00:01
     |████████████████████████████████| 194kB 24.4MB/s eta 0:00:01
     |████████████████████████████████| 143kB 25.9MB/s eta 0:00:01
     |████████████████████████████████| 51kB 9.2MB/s  eta 0:00:01
     |████████████████████████████████| 296kB 25.5MB/s eta 0:00:01
     |████████████████████████████████| 184kB 41.2MB/s eta 0:00:01
     |████████████████████████████████| 235kB 41.5MB/s eta 0:00:01
     |████████████████████████████████| 71kB 4.5MB/s  eta 0:00:01
     |████████████████████████████████| 5.6MB 32.3MB/s eta 0:00:01
     |████████████████████████████████| 92kB 14.7MB/s eta 0:00:01
     |████████████████████████████████| 143kB 58.3MB/s eta 0:00:01
     |████████████████████████████████| 71kB 14.3MB/s eta 0:00:01
     |████████████████████████████████| 163kB 62.7MB/s eta 0:00:01
     |████████████████████████████████| 122kB 36.3MB/s eta 0:00:01
ERROR: jupyterlab-server 2.25.2 has requirement babel>=2.10, but yo

In [ ]:
from google.cloud import monitoring_v3
import time

project_id = "fdc-development-408605"
client = monitoring_v3.MetricServiceClient()
project_name = f"projects/{project_id}"

now = time.time()
seconds = int(now)
nanos = int((now - seconds) * 10 ** 9)
interval = monitoring_v3.TimeInterval(
    {
        "end_time": {"seconds": seconds, "nanos": nanos},
        "start_time": {"seconds": (seconds - 3600), "nanos": nanos},
    }
)
aggregation = monitoring_v3.Aggregation(
    {
        "alignment_period": {"seconds": 1200},  # 20 minutes
        "per_series_aligner": monitoring_v3.Aggregation.Aligner.ALIGN_MEAN,
        "cross_series_reducer": monitoring_v3.Aggregation.Reducer.REDUCE_MEAN,
        "group_by_fields": ["resource.zone"],
    }
)

results = client.list_time_series(
    request={
        "name": project_name,
        "filter": 'metric.type = "compute.googleapis.com/instance/cpu/utilization"',
        "interval": interval,
        "view": monitoring_v3.ListTimeSeriesRequest.TimeSeriesView.FULL,
        "aggregation": aggregation,
    }
)
for result in results:
    print(result)


In [1]:
from google.cloud import monitoring_v3
import time
project_id = "fdc-development-408605"
pod_name = "notebooks-worker-797d95f49-cv92v"
client = monitoring_v3.MetricServiceClient()
project_name = f"projects/{project_id}"

now = time.time()
seconds = int(now)
nanos = int((now - seconds) * 10 ** 9)
interval = monitoring_v3.TimeInterval(
    {
        "end_time": {"seconds": seconds, "nanos": nanos},
        "start_time": {"seconds": (seconds - 3600), "nanos": nanos},
    }
)
aggregation = monitoring_v3.Aggregation(
    {
        "alignment_period": {"seconds": 1200},  # 20 minutes
        "per_series_aligner": monitoring_v3.Aggregation.Aligner.ALIGN_MEAN,
        "cross_series_reducer": monitoring_v3.Aggregation.Reducer.REDUCE_MEAN,
    }
)

filter_str = f'resource.type="k8s_pod" AND resource.labels.pod_name="{pod_name}" AND metric.type="kubernetes.io/container/cpu/usage_time"'
print('1')
results = client.list_time_series(
    request={
        "name": project_name,
        "filter": filter_str,
        "interval": interval,
        "view": monitoring_v3.ListTimeSeriesRequest.TimeSeriesView.FULL,
        "aggregation": aggregation,
    }
)
print('2')
for result in results:
    print(result)
print('3')

1


RetryError: Timeout of 90.0s exceeded, last exception: 503 Getting metadata from plugin failed with error: ('Failed to retrieve http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/insight-cloud-monitoring-test@fdc-development-408605.iam.gserviceaccount.com/token?scopes=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform%2Chttps%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email from the Google Compute Engine metadata service. Status: 403 Response:\nb"Unable to generate access token; IAM returned 403 Forbidden: Permission \'iam.serviceAccounts.getAccessToken\' denied on resource (or it may not exist).\\nThis error could be caused by a missing IAM policy binding on the target IAM service account.\\nFor more information, refer to the Workload Identity documentation:\\n\\thttps://cloud.google.com/kubernetes-engine/docs/how-to/workload-identity#authenticating_to\\n\\n"', <google.auth.transport.requests._Response object at 0x7d87601acee0>)

In [2]:
!curl -H "Metadata-Flavor: Google" http://169.254.169.254/computeMetadata/v1/instance/service-accounts/default/email

insight-cloud-monitoring-test@fdc-development-408605.iam.gserviceaccount.com

In [3]:
!curl -H "Metadata-Flavor: Google" http://169.254.169.254/computeMetadata/v1/instance/service-accounts/default/token

Unable to generate access token; IAM returned 403 Forbidden: Permission 'iam.serviceAccounts.getAccessToken' denied on resource (or it may not exist).
This error could be caused by a missing IAM policy binding on the target IAM service account.
For more information, refer to the Workload Identity documentation:
	https://cloud.google.com/kubernetes-engine/docs/how-to/workload-identity#authenticating_to

